# Example for sampling mRS score from many probability distributions

Example of sampling from many different probability distributions for many patients simultaneously.

Method:
+ Assign a different mRS probability distribution to each patient. Here use the age vs mRS probabilities to do that.
+ Randomly generate some "x" from 0 to 1, with one value for each patient.
+ Use the x values to sample from the mRS distribution that matches that patient. Use pd.DataFrame and the apply() function.

## Code setup

In [1]:
import numpy as np
import pandas as pd
import os

## Set up probability distributions

Expect to have a dataframe with different "x" and different probability distributions.

Load in a dataframe with more variety of probability distributions:

In [2]:
p = os.path.join('summary_data', 'combo_age_disability.csv')

df_age_dis_full = pd.read_csv(p, index_col=0)

Make sure each column sums to 1:

In [3]:
df_probs = (df_age_dis_full / df_age_dis_full.sum(axis='rows'))

In [4]:
df_probs.sum(axis='rows')

37.5    1.0
42.5    1.0
47.5    1.0
52.5    1.0
57.5    1.0
62.5    1.0
67.5    1.0
72.5    1.0
77.5    1.0
82.5    1.0
87.5    1.0
92.5    1.0
dtype: float64

In [5]:
df_probs

,37.5,42.5,47.5,52.5,57.5,62.5,67.5,72.5,77.5,82.5,87.5,92.5
prior_disability,,,,,,,,,,,,
0,0.850438,0.848158,0.817359,0.790403,0.750952,0.705610,0.646923,0.588105,0.512663,0.395572,0.284265,0.153814
1,0.083032,0.087675,0.104484,0.116734,0.133300,0.158043,0.181988,0.204594,0.222883,0.228336,0.206364,0.149387
2,0.030428,0.027954,0.041958,0.045118,0.056963,0.065092,0.079599,0.088679,0.106997,0.141619,0.160285,0.159168
3,0.024755,0.021601,0.024270,0.032705,0.037258,0.046989,0.059342,0.075964,0.099770,0.152119,0.215771,0.303923
4,0.009799,0.013342,0.009050,0.011936,0.017884,0.019643,0.026203,0.033798,0.046112,0.066730,0.109943,0.190981
5,0.001547,0.001271,0.002879,0.003103,0.003643,0.004622,0.005945,0.008860,0.011576,0.015623,0.023370,0.042726


Convert probabilities to cumulative probabilities:

In [6]:
df_probs = np.cumsum(df_probs, axis='rows')
rename_dict = dict(zip(range(0, 6), [f'mrs<={x}' for x in range(0, 6)]))
df_probs = df_probs.rename(index=rename_dict)

In [7]:
df_probs

,37.5,42.5,47.5,52.5,57.5,62.5,67.5,72.5,77.5,82.5,87.5,92.5
prior_disability,,,,,,,,,,,,
mrs<=0,0.850438,0.848158,0.817359,0.790403,0.750952,0.705610,0.646923,0.588105,0.512663,0.395572,0.284265,0.153814
mrs<=1,0.933471,0.935832,0.921843,0.907138,0.884252,0.863654,0.828911,0.792699,0.735546,0.623909,0.490630,0.303202
mrs<=2,0.963899,0.963787,0.963801,0.952256,0.941215,0.928746,0.908510,0.881378,0.842543,0.765528,0.650915,0.462370
mrs<=3,0.988654,0.985388,0.988071,0.984961,0.978473,0.975735,0.967852,0.957342,0.942313,0.917647,0.866686,0.766293
mrs<=4,0.998453,0.998729,0.997121,0.996897,0.996357,0.995378,0.994055,0.991140,0.988424,0.984377,0.976630,0.957274
mrs<=5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Generate "x" values

In [8]:
n_patients = 30

In [9]:
np.random.seed(42)

x_values = np.random.uniform(size=n_patients)
# Round:
x_values = np.round(x_values, 5)

x_values

array([0.37454, 0.95071, 0.73199, 0.59866, 0.15602, 0.15599, 0.05808,
       0.86618, 0.60112, 0.70807, 0.02058, 0.96991, 0.83244, 0.21234,
       0.18182, 0.1834 , 0.30424, 0.52476, 0.43195, 0.29123, 0.61185,
       0.13949, 0.29214, 0.36636, 0.45607, 0.78518, 0.19967, 0.51423,
       0.59241, 0.04645])

## Gather data

Set up a dataframe with different probability distributions on each line.

Just select ages at random rather than weighting by likeliness of age in the stroke data.

In [10]:
np.random.seed(42)

df_multi = pd.DataFrame()

df_multi['x'] = x_values  # from before
# Pick out ages at random:
df_multi['age'] = np.random.choice(df_probs.columns, size=len(df_multi))
# Pick out probability distributions that match ages:
df_multi = pd.merge(df_multi, df_probs.T, left_on='age', right_index=True, how='left')

Should see that each patient has a different "x" value. The patients with the same age should have the same probability distribution.

View the results, sorted by age and then by "x":

In [11]:
df_multi.sort_values(['age', 'x'])

,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5
26,0.19967,37.5,0.850438,0.933471,0.963899,0.988654,0.998453,1.0
19,0.29123,42.5,0.848158,0.935832,0.963787,0.985388,0.998729,1.0
23,0.36636,42.5,0.848158,0.935832,0.963787,0.985388,0.998729,1.0
16,0.30424,47.5,0.817359,0.921843,0.963801,0.988071,0.997121,1.0
7,0.86618,47.5,0.817359,0.921843,0.963801,0.988071,0.997121,1.0
13,0.21234,52.5,0.790403,0.907138,0.952256,0.984961,0.996897,1.0
1,0.95071,52.5,0.790403,0.907138,0.952256,0.984961,0.996897,1.0
4,0.15602,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0
18,0.43195,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0
25,0.78518,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0


## Sample mRS distributions

Set up a function to sample the probability distribution from each row based on the "x" value:

In [12]:
def pick_out_mrs(x_values, cprobs):
    x_inds = np.digitize(x_values, cprobs, right=True)
    return x_inds

Run the function on all rows of the dataframe using apply():

In [13]:
# Notes on the below line of code.
# df_multi        # dataframe
# .apply()        # tell Pandas to run function on each row separately
# lambda r:       # picks out one row at a time and names it "r"
# pick_out_mrs()  # the name of the function we're running
# r['x']          # row r, column "x" is first arg for pick_out_mrs()
# [r[f'mrs<={m}'] for m in range(6)]  # row r, mRS columns are put in a list.
#                                     # list is second arg for pick_out_mrs()
# axis=1          # specifies we want it row-by-row.

df_multi['chosen_mrs'] = df_multi.apply(
    lambda r: pick_out_mrs(
        r['x'], [r[f'mrs<={m}'] for m in range(6)]
    ), axis=1)

Check first few results:

In [14]:
df_multi.head(3)

,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
0,0.37454,67.5,0.646923,0.828911,0.908510,0.967852,0.994055,1.0,0
1,0.95071,52.5,0.790403,0.907138,0.952256,0.984961,0.996897,1.0,2
2,0.73199,87.5,0.284265,0.490630,0.650915,0.866686,0.976630,1.0,3


Check all results by grouping patients by age (same probability distributions):

In [15]:
for age, df in df_multi.groupby('age'):
    display(df.sort_values('x'))

,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
26,0.19967,37.5,0.850438,0.933471,0.963899,0.988654,0.998453,1.0,0


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
19,0.29123,42.5,0.848158,0.935832,0.963787,0.985388,0.998729,1.0,0
23,0.36636,42.5,0.848158,0.935832,0.963787,0.985388,0.998729,1.0,0


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
16,0.30424,47.5,0.817359,0.921843,0.963801,0.988071,0.997121,1.0,0
7,0.86618,47.5,0.817359,0.921843,0.963801,0.988071,0.997121,1.0,1


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
13,0.21234,52.5,0.790403,0.907138,0.952256,0.984961,0.996897,1.0,0
1,0.95071,52.5,0.790403,0.907138,0.952256,0.984961,0.996897,1.0,2


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
4,0.15602,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0,0
18,0.43195,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0,0
25,0.78518,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0,1
12,0.83244,57.5,0.750952,0.884252,0.941215,0.978473,0.996357,1.0,1


...

,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
5,0.15599,67.5,0.646923,0.828911,0.90851,0.967852,0.994055,1.0,0
0,0.37454,67.5,0.646923,0.828911,0.90851,0.967852,0.994055,1.0,0
8,0.60112,67.5,0.646923,0.828911,0.90851,0.967852,0.994055,1.0,0


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
14,0.18182,72.5,0.588105,0.792699,0.881378,0.957342,0.99114,1.0,0
15,0.18340,72.5,0.588105,0.792699,0.881378,0.957342,0.99114,1.0,0
3,0.59866,72.5,0.588105,0.792699,0.881378,0.957342,0.99114,1.0,1
20,0.61185,72.5,0.588105,0.792699,0.881378,0.957342,0.99114,1.0,1
11,0.96991,72.5,0.588105,0.792699,0.881378,0.957342,0.99114,1.0,4


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
6,0.05808,82.5,0.395572,0.623909,0.765528,0.917647,0.984377,1.0,0
28,0.59241,82.5,0.395572,0.623909,0.765528,0.917647,0.984377,1.0,1


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
10,0.02058,87.5,0.284265,0.49063,0.650915,0.866686,0.97663,1.0,0
9,0.70807,87.5,0.284265,0.49063,0.650915,0.866686,0.97663,1.0,3
2,0.73199,87.5,0.284265,0.49063,0.650915,0.866686,0.97663,1.0,3


,x,age,mrs<=0,mrs<=1,mrs<=2,mrs<=3,mrs<=4,mrs<=5,chosen_mrs
21,0.13949,92.5,0.153814,0.303202,0.46237,0.766293,0.957274,1.0,0
24,0.45607,92.5,0.153814,0.303202,0.46237,0.766293,0.957274,1.0,2
27,0.51423,92.5,0.153814,0.303202,0.46237,0.766293,0.957274,1.0,3
